# Data Preprocessing: Exercise

This exercise covers the cleaning and preprocessing of the **Boston Housing Dataset**, a dataset containing various information about housing in the Boston area.

The dataset contains the following variables:

| Column | Description |
|--------|-------------|
| `CRIM` | Per capita crime rate by town |
| `ZN` | Proportion of residential land zoned for lots over 25,000 sq.ft. |
| `INDUS` | Proportion of non-retail business acres per town |
| `CHAS` | Dummy variable indicating proximity to the Charles River |
| `NOX` | Nitric oxide concentration (parts per 10 million) |
| `RM` | Average number of rooms per dwelling |
| `AGE` | Proportion of owner-occupied units built prior to 1940 |
| `DIS` | Weighted distances to five Boston employment centres |
| `RAD` | Index of accessibility to radial highways |
| `TAX` | Full-value property tax rate per $10,000 |
| `PTRATIO` | Pupil-teacher ratio by town |
| `B` | 1000(Bk - 0.63)² where Bk is the proportion of Black residents by town |
| `LSTAT` | Percentage of lower-status population |
| `PRICE` | Median value of owner-occupied homes in $1,000s |

**Tasks:**

1. The number of rows and columns in the dataset is verified
2. The data type of each variable is checked
3. The number of missing values per column is reported
4. Columns with more than 30% missing values are dropped
5. Rows with more than 25% missing values are dropped
6. All rows where `PRICE` is missing are dropped
7. Remaining missing quantitative values are imputed with the column mean
8. Qualitative variables are encoded
9. Remaining missing qualitative values are imputed with the column mode
10. Features are standardized
11. The cleaned dataframe is saved as a TSV file named `housing_clean.tsv`

In [39]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# load the raw (dirty) dataset — index column contains row numbers, not features
df = pd.read_csv('datasets/housing_dirty.csv', index_col=0)

## Step 1 - Rows and Columns

The first step in any data cleaning pipeline is to understand the **size and shape** of the dataset. This confirms that the file loaded correctly and gives a baseline count of observations and features before any rows or columns are dropped.

In [40]:
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
df.head()

Rows: 506, Columns: 14


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,PRICE
0.0,LOW,18.0,2.31,NO,538.0,6575.0,65.2,4.0900,1.0,296.0,15.3,396.90,4.98,24.0
1.0,LOW,0.0,7.07,NO,469.0,6421.0,78.9,4.9671,2.0,242.0,17.8,396.90,9.14,21.6
2.0,LOW,0.0,7.07,NO,469.0,7185.0,61.1,4.9671,2.0,242.0,17.8,392.83,4.03,34.7
3.0,LOW,0.0,2.18,NO,458.0,6998.0,45.8,6.0622,3.0,222.0,18.7,394.63,2.94,33.4
4.0,LOW,0.0,2.18,NO,458.0,7147.0,54.2,6.0622,3.0,222.0,18.7,396.90,NaN,36.2


## Step 2 - Variable Types

Knowing each column's **dtype** is essential because it determines which preprocessing operations are applicable:
- `float64` / `int64` → **quantitative** variables: can be imputed with mean, standardized, used directly in regression.
- `object` → **qualitative** (categorical) variables: must be encoded into numbers before a model can use them, and should be imputed with the mode rather than the mean.

In [41]:
df.dtypes

CRIM        object
ZN         float64
INDUS      float64
CHAS        object
NOX        float64
RM         float64
AGE        float64
DIS        float64
RAD        float64
TAX        float64
PTRATIO    float64
B          float64
LSTAT      float64
PRICE      float64
dtype: object

## Step 3 - Missing Values per Column

Missing values must be identified before deciding how to handle them. The strategy depends on the **proportion** of missing data:
- A column with very few missing values can be safely imputed.
- A column with too many missing values carries very little information and is better dropped entirely.
- A row missing most of its values is similarly uninformative and should be removed.

In [42]:
missing = df.isnull().sum()
missing_pct = missing / len(df) * 100
pd.DataFrame({'missing': missing, 'missing_%': missing_pct.round(2)})

,missing,missing_%
CRIM,0,0.00
ZN,2,0.40
INDUS,3,0.59
CHAS,0,0.00
NOX,7,1.38
RM,5,0.99
AGE,4,0.79
DIS,5,0.99
RAD,3,0.59
TAX,2,0.40


## Step 4 - Drop Columns with > 30% Missing Values

A threshold of **30%** is used: if more than 30% of a column's values are missing, imputation would introduce too much artificial data and could distort the distribution. The column is dropped instead.

> **Result:** `LSTAT` (39.3% missing) is dropped, reducing the feature count from 14 to 13.

In [43]:
threshold_col = 0.30

# identify columns where the missing percentage exceeds the threshold
cols_to_drop = missing_pct[missing_pct > threshold_col * 100].index.tolist()
print(f'Columns dropped (>{threshold_col*100:.0f}% missing): {cols_to_drop}')

df.drop(columns=cols_to_drop, inplace=True)
print(f'Shape after drop: {df.shape}')

Columns dropped (>30% missing): ['LSTAT']
Shape after drop: (506, 13)


## Step 5 - Drop Rows with > 25% Missing Values

A row with more than 25% of its feature values missing provides very little useful information and could introduce bias if imputed. A threshold of **25%** is applied across the remaining 13 columns (i.e., rows with more than 3 missing values are dropped).

> **Result:** 5 rows are removed.

In [44]:
threshold_row = 0.25

# compute the fraction of missing values for each row (after column drop)
row_missing_pct = df.isnull().sum(axis=1) / df.shape[1]
rows_to_drop = row_missing_pct[row_missing_pct > threshold_row].index

print(f'Rows dropped (>{threshold_row*100:.0f}% missing): {len(rows_to_drop)}')
df.drop(index=rows_to_drop, inplace=True)
print(f'Shape after drop: {df.shape}')

Rows dropped (>25% missing): 5
Shape after drop: (501, 13)


## Step 6 - Drop Rows where PRICE is Missing

`PRICE` is the **target variable** (what the model is trying to predict). A row with a missing target value cannot be used for training a supervised model — there is nothing to learn from it. These rows are dropped unconditionally, regardless of the threshold applied in step 5.

> **Result:** 4 additional rows are removed.

In [45]:
before = len(df)
df.dropna(subset=['PRICE'], inplace=True)
print(f'Rows dropped (PRICE missing): {before - len(df)}')
print(f'Shape after drop: {df.shape}')

Rows dropped (PRICE missing): 4
Shape after drop: (497, 13)


## Step 7 - Mean Imputation for Quantitative Variables

For numerical columns with a small number of missing values, the missing entries are replaced with the **column mean**. This is a simple and commonly used strategy that preserves the overall distribution of the column without introducing bias (assuming values are missing at random).

Mean imputation is appropriate here because the proportion of missing values per column is low (< 2%). For larger proportions, more sophisticated methods (e.g. KNN imputation, regression imputation) would be preferable.

In [46]:
num_cols = df.select_dtypes(include='number').columns  # only numeric columns

for col in num_cols:
    n = df[col].isnull().sum()
    if n > 0:
        df.fillna({col: df[col].mean()}, inplace=True)
        print(f'  {col}: imputed {n} values with mean ({df[col].mean():.4f})')

print(f'\nRemaining missing (quantitative): {df[num_cols].isnull().sum().sum()}')

  INDUS: imputed 1 values with mean (11.1562)
  NOX: imputed 4 values with mean (463.2787)
  DIS: imputed 1 values with mean (384.3893)
  PTRATIO: imputed 3 values with mean (18.4690)
  B: imputed 1 values with mean (356.6370)

Remaining missing (quantitative): 0


## Step 8 - Encode Qualitative Variables

Machine learning models require all input features to be **numeric**. The two categorical columns are encoded as follows:

- **`CRIM`** (crime rate category): encoded as an **ordinal** variable because the categories have a natural order from low to high crime rate. Ordinal encoding preserves this order information.
  - `LOW` → 0, `MODERATE` → 1, `HIGH` → 2, `VERY HIGH` → 3
- **`CHAS`** (proximity to Charles River): encoded as a **binary** variable (0/1) since there are only two categories with no inherent order. This is equivalent to one-hot encoding for a binary feature.
  - `NO` → 0, `YES` → 1

In [47]:
# ordinal encoding for CRIM: order reflects increasing crime severity
crim_order = {'LOW': 0, 'MODERATE': 1, 'HIGH': 2, 'VERY HIGH': 3}
df['CRIM'] = df['CRIM'].map(crim_order)

# binary encoding for CHAS: 1 = borders the Charles River, 0 = does not
df['CHAS'] = df['CHAS'].map({'NO': 0, 'YES': 1})

print('Encoded — CRIM value counts:')
print(df['CRIM'].value_counts().sort_index())
print('\nEncoded — CHAS value counts:')
print(df['CHAS'].value_counts().sort_index())

Encoded — CRIM value counts:
CRIM
0    127
1    120
2    126
3    124
Name: count, dtype: int64

Encoded — CHAS value counts:
CHAS
0    463
1     34
Name: count, dtype: int64


## Step 9 - Mode Imputation for Qualitative Variables

For categorical columns with missing values, the **mode** (most frequent value) is used as the imputation strategy. Using the mean is not meaningful for categories; the mode preserves the most common class and is the standard approach for categorical imputation.

In this dataset, after encoding in Step 8 both `CRIM` and `CHAS` have no remaining `NaN` values, so no imputation is performed. The check is included for completeness and robustness.

In [48]:
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    n = df[col].isnull().sum()
    if n > 0:
        mode_val = df[col].mode()[0]
        df.fillna({col: mode_val}, inplace=True)
        print(f'  {col}: imputed {n} values with mode ({mode_val})')

print(f'Total remaining missing values: {df.isnull().sum().sum()}')

Total remaining missing values: 0


## Step 10 - Standardization

`StandardScaler` transforms each feature to have **mean = 0** and **standard deviation = 1** (Z-score normalization):

$$z = \frac{x - \mu}{\sigma}$$

Standardization is important for:
- **Distance-based models** (KNN, SVM): features with large ranges would otherwise dominate.
- **Regularized models** (Ridge, Lasso): the penalty is applied to coefficients, which depend on feature scale.
- **Gradient-based optimization**: convergence is faster and more stable when features are on the same scale.

Note: `PRICE` is excluded from standardization since it is the target variable, not an input feature. Its raw values ($1,000s) are preserved for interpretability.

In [49]:
scaler = StandardScaler()

# exclude PRICE (target) from scaling — scale only input features
features = [c for c in df.columns if c != 'PRICE']
df[features] = scaler.fit_transform(df[features])

# after standardization: mean ≈ 0, std ≈ 1 for all feature columns; PRICE is unchanged
df.describe().round(3)

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,PRICE
count,497.000,497.000,497.000,497.000,497.000,497.000,497.000,497.000,497.000,497.000,497.000,497.000,497.000
mean,0.000,0.000,-0.000,-0.000,-0.000,-0.000,-0.000,-0.000,-0.000,0.000,0.000,-0.000,22.494
std,1.001,1.001,1.001,1.001,1.001,1.001,1.001,1.001,1.001,1.001,1.001,1.001,9.188
min,-1.334,-0.491,-1.557,-0.271,-2.070,-2.798,-2.320,-0.300,-0.984,-1.313,-2.734,-3.888,5.000
25%,-1.334,-0.491,-0.877,-0.271,-0.158,0.060,-0.851,-0.299,-0.642,-0.770,-0.498,0.204,16.800
50%,0.448,-0.491,-0.213,-0.271,0.160,0.250,0.314,-0.298,-0.527,-0.469,0.294,0.380,21.200
75%,0.448,0.042,1.011,-0.271,0.634,0.457,0.909,-0.296,1.642,1.514,0.806,0.432,25.000
max,1.339,3.772,2.414,3.690,1.823,1.542,1.118,6.234,1.642,1.780,1.645,0.439,50.000


## Step 11 - Save Cleaned Dataset

The cleaned dataframe is saved as a **TSV** (Tab-Separated Values) file. TSV is preferred over CSV when column values may contain commas, which is common with text data. The file can be loaded in future notebooks with `pd.read_csv(..., sep='\t')`.

In [50]:
output_path = 'datasets/housing_clean.tsv'
df.to_csv(output_path, sep='\t')
print(f'Saved to {output_path} — shape: {df.shape}')

Saved to datasets/housing_clean.tsv — shape: (497, 13)


## Conclusions

The raw dataset (506 rows × 14 columns) has been cleaned and is ready for modelling. The table below summarizes each decision made during the pipeline:

| Step | Action | Outcome |
|------|--------|---------|
| Drop columns | `LSTAT` dropped (39.3% missing > 30% threshold) | 14 → 13 columns |
| Drop rows | 5 rows dropped (> 25% of features missing) | 506 → 501 rows |
| Drop target rows | 4 rows dropped (`PRICE` missing) | 501 → 497 rows |
| Mean imputation | 5 numeric columns imputed (< 2% missing each) | 0 remaining numeric NaN |
| Ordinal encoding | `CRIM`: LOW/MODERATE/HIGH/VERY HIGH → 0/1/2/3 | Preserves natural order |
| Binary encoding | `CHAS`: NO/YES → 0/1 | Two-class variable |
| Mode imputation | No action needed (no categorical NaN remaining) | - |
| Standardization | All features (except `PRICE`) scaled to mean=0, std=1 | ML-ready feature matrix |

**Final dataset: 497 rows × 13 columns** (12 features + 1 target).